# Практикум 2. Поиск нетипичных операций: от правил к гибриду (Google Colab)
**После блокнота вы умеете:** (1) строить признаки под конкретные схемы нарушений; (2) сравнивать правила, робастный z, Isolation Forest, LOF и автоэнкодер на одном наборе по precision@k и полноте по типам закладок; (3) собирать гибридную очередь проверки (правила + ML); (4) получать причины срабатывания для каждого алерта; (5) проверять устойчивость и объём истории.
**Данные:** синтетическая книга проводок после очистки по П1 (с заложенными типами нарушений A–E и ground truth) и открытый журнал Loghub HDFS_2k (сессии по блокам, без разметки).
**Артефакты:** `p2_metrics.json`, `anomaly_queue.csv` (читает П3). Запуск: Среда выполнения → Выполнить всё.

In [ ]:
import numpy as np, pandas as pd, json, re, urllib.request
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score
SEED = 42; rng = np.random.default_rng(SEED); pd.options.display.float_format = '{:,.3f}'.format
ACC = ['4010','4020','5010','5020','6010','7010']; USERS = [f'u{i:02d}' for i in range(30)]
HOME = {u: rng.choice(ACC, 2, replace=False) for u in USERS}          # у каждого автора два «своих» счёта
OFF = dict(zip(ACC, [0, -.3, .2, .5, -.1, .8]))                       # масштаб сумм зависит от счёта

## 0. Данные: типы нарушений A–E
A — круглые крупные, B — дробление под лимит 10 000, C — ночь + редкий автор (эти три знают правила из П1). **D — чужой счёт** (автор пишет в не свой счёт, сумма крупная для этого счёта), **E — всплеск** (автор за день делает 20 лишних проводок). D и E правила не видят.

In [ ]:
def make_period(n, start, mu, id0):
    d = pd.DataFrame({'entry_id': np.arange(n)+id0}); d['user'] = rng.choice(USERS, n)
    d['account'] = [rng.choice(HOME[u]) if rng.random() < .9 else rng.choice(ACC) for u in d.user]
    day = rng.integers(0, 90, n); h = rng.choice(24, n, p=np.r_[np.full(7,.002),np.full(11,.085),np.full(6,.0095)]/(.002*7+.085*11+.0095*6))
    d['created_ts'] = pd.Timestamp(start)+pd.to_timedelta(day,'D')+pd.to_timedelta(h*3600+rng.integers(0,3600,n),'s')
    d['post_date'] = d.created_ts.dt.normalize()+pd.to_timedelta(rng.poisson(1.2,n),'D')
    d['amount'] = np.round(rng.lognormal(mu+d.account.map(OFF),1.0),2); d['atype'] = ''
    def mark(ix,t): d.loc[ix,'atype'] = t
    free = list(rng.permutation(n)); take = lambda k: [free.pop() for _ in range(k)]
    a = take(50); d.loc[a,'amount'] = rng.choice([50000,100000,250000],50); mark(a,'A')
    b = take(50); d.loc[b,'amount'] = np.round(rng.uniform(9500,9999.99,50),2); mark(b,'B')
    c = take(40); d.loc[c,'user'] = 'u_rare'; d.loc[c,'created_ts'] = d.loc[c,'created_ts'].dt.normalize()+pd.Timedelta(hours=2,minutes=17); mark(c,'C')
    e = take(50)
    for i in e:
        u = d.at[i,'user']; d.at[i,'account'] = rng.choice([a_ for a_ in ACC if a_ not in HOME[u]])
        d.at[i,'amount'] = round(float(np.exp(mu+OFF[d.at[i,'account']]+1.6+rng.normal(0,.3))),2)
    mark(e,'D')
    for _ in range(4):                                                  # E: 4 события по 20 проводок
        u = rng.choice(USERS); day0 = pd.Timestamp(start)+pd.Timedelta(days=int(rng.integers(5,85))); ix = take(20)
        d.loc[ix,'user'] = u; d.loc[ix,'created_ts'] = day0+pd.to_timedelta(rng.integers(9*3600,17*3600,20),'s')
        d.loc[ix,'post_date'] = day0; d.loc[ix,'amount'] = np.round(rng.uniform(2000,6000,20),2); mark(ix,'E')
    d['is_anomaly'] = (d.atype != '').astype(int); return d
q1 = make_period(20000,'2026-01-01',7.0,0); q2 = make_period(20000,'2026-04-01',7.25,10**6)
print(q2.atype.value_counts().drop('').to_dict(), '| доля закладок', q2.is_anomaly.mean().round(4))

## 1. Признаки под схемы нарушений
Частоты и робастные статистики считаем **по эталону (Q1)**; признак «проводок автора за день» берём из самого периода — он наблюдаем в момент проверки.

In [ ]:
def feats(d, ref):
    f = pd.DataFrame(index=d.index); f['log_amt'] = np.log1p(d.amount); h = d.created_ts.dt.hour
    f['night'] = ((h<6)|(h>=22)).astype(int); f['weekend'] = (d.created_ts.dt.dayofweek>=5).astype(int)
    f['round'] = ((d.amount%1000==0)&(d.amount>=10000)).astype(int); f['near_limit'] = d.amount.between(9500,9999.99).astype(int)
    f['user_freq'] = d.user.map(ref.user.value_counts(normalize=True)).fillna(0)
    share = (ref.groupby(['user','account']).size()/ref.groupby('user').size()).rename('s')
    f['pair_share'] = pd.MultiIndex.from_frame(d[['user','account']]).map(lambda k: share.get(k, 0.0))      # доля счёта в проводках автора
    lr = np.log1p(ref.amount); med = lr.groupby(ref.account).median(); mad = (lr-ref.account.map(med)).abs().groupby(ref.account).median()
    f['acc_z'] = 0.6745*(f.log_amt-d.account.map(med))/d.account.map(mad)                              # крупно для своего счёта
    f['user_day_cnt'] = d.groupby(['user', d.created_ts.dt.date]).entry_id.transform('count')
    f['lag'] = (d.post_date-d.created_ts.dt.normalize()).dt.days; return f
F1, F2 = feats(q1,q1), feats(q2,q1); y = q2.is_anomaly.values; types = q2.atype.values
print(F2.describe().T[['mean','min','max']])

## 2. Пять методов на одном наборе
Обучение — Q1, оценка — Q2. Рейтинг: больше = подозрительнее. `contamination` не задаём — порог определяет ёмкость проверки.

In [ ]:
def rank01(s): return pd.Series(s).rank(pct=True).values
S = {}
S['правила'] = (F2['round']+F2.near_limit+((F2.night==1)&(F2.user_freq<.01))).values + 1e-3*F2.log_amt.values
S['робастный z по счёту'] = np.abs(F2.acc_z.values)
iso = IsolationForest(n_estimators=300, max_samples=256, random_state=SEED, n_jobs=-1).fit(F1); S['Isolation Forest'] = -iso.score_samples(F2)
sc = RobustScaler().fit(F1); X1, X2 = np.clip(sc.transform(F1),-10,10), np.clip(sc.transform(F2),-10,10)
S['LOF'] = -LocalOutlierFactor(n_neighbors=35, novelty=True, n_jobs=-1).fit(X1).score_samples(X2)
ae = MLPRegressor(hidden_layer_sizes=(8,3,8), max_iter=300, early_stopping=True, random_state=SEED).fit(X1, X1)
S['автоэнкодер'] = ((ae.predict(X2)-X2)**2).mean(axis=1)
SUB = {'чужой счёт': ['pair_share','acc_z','log_amt'], 'всплеск': ['user_day_cnt','user_freq','log_amt'], 'режим работы': ['night','weekend','lag','user_freq']}
sub_sc = [rank01(-IsolationForest(n_estimators=200, max_samples=256, random_state=SEED).fit(F1[c]).score_samples(F2[c])) for c in SUB.values()]
S['IF по схемам (max рангов)'] = np.max(sub_sc, axis=0)                  # отдельный IF на каждую схему нарушения
S['среднее рангов IF+LOF+AE'] = (rank01(S['Isolation Forest'])+rank01(S['LOF'])+rank01(S['автоэнкодер']))/3
S['max рангов IF+AE'] = np.maximum(rank01(S['Isolation Forest']), rank01(S['автоэнкодер']))
hit = ((F2['round']==1)|(F2.near_limit==1)|((F2.night==1)&(F2.user_freq<.01))).values
S['гибрид'] = hit*10 + S['IF по схемам (max рангов)']                  # сначала срабатывания правил, остаток — IF по схемам

In [ ]:
def prec_k(s,k): return float(y[np.argsort(-s)[:k]].mean())
def recall_by_type(s,k):
    top = set(np.argsort(-s)[:k]); return {t: float(np.mean([i in top for i in np.where(types==t)[0]])) for t in 'ABCDE'}
K = [50,200,500]; R = {m: {**{f'P@{k}': prec_k(s,k) for k in K}, 'PR-AUC': float(average_precision_score(y,s))} for m,s in S.items()}
RT = pd.DataFrame({m: recall_by_type(s,500) for m,s in S.items()}).T; RT['все'] = [float(y[np.argsort(-s)[:500]].sum()/y.sum()) for s in S.values()]
print(pd.DataFrame(R).T); print('\nполнота@500 по типам'); print(RT)

## 3. Устойчивость: сиды и объём истории

In [ ]:
P200 = []
for sd in range(5):
    m = IsolationForest(n_estimators=300, max_samples=256, random_state=sd, n_jobs=-1).fit(F1); P200.append(prec_k(-m.score_samples(F2),200))
print('IF P@200 по 5 сидам: среднее %.3f, σ %.3f' % (np.mean(P200), np.std(P200)))
SZ = {}
for n in [2000,5000,10000,20000]:
    ix = rng.choice(len(F1), n, replace=False); m = IsolationForest(n_estimators=300, max_samples=min(256,n), random_state=SEED, n_jobs=-1).fit(F1.iloc[ix]); SZ[n] = prec_k(-m.score_samples(F2),200)
print('P@200 от размера эталона:', {k: round(v,3) for k,v in SZ.items()})

## 4. Очередь проверки с причинами срабатывания
Причина — признак с наибольшим робастным отклонением от эталона: `z = (x − медиана) / MAD`.

In [ ]:
def reasons(F, ref_F, idx, top=3):
    med = ref_F.median(); mad = (ref_F-med).abs().median().replace(0, np.nan)
    Z = ((F.iloc[idx]-med)/mad).replace([np.inf,-np.inf], np.nan).fillna(0)
    return [', '.join(f'{c} ({z:+.1f})' for c, z in row.reindex(row.abs().sort_values(ascending=False).index)[:top].items()) for _, row in Z.iterrows()]
k = 100; top = np.argsort(-S['гибрид'])[:k]
Q = q2.iloc[top][['entry_id','user','account','amount','created_ts','atype']].copy(); Q['score'] = S['гибрид'][top]; Q['reasons'] = reasons(F2, F1, top)
print('precision очереди k=%d: %.3f' % (k, Q.atype.ne('').mean())); print(Q.head(5)[['entry_id','user','account','amount','reasons']].to_string())
Q.drop(columns='atype').to_csv('anomaly_queue.csv', index=False)
for t in 'DE': print(t, Q[Q.atype==t].reasons.head(1).tolist(), '| в очереди:', int((Q.atype==t).sum()))

In [ ]:
def quota_queue(scores, k):                                  # по кругу берём лучшие из списков каждой схемы
    orders = [list(np.argsort(-s)) for s in scores]; seen, i = [], 0
    while len(seen) < k:
        for o in orders:
            if o[i] not in seen: seen.append(o[i])
            if len(seen) == k: break
        i += 1
    return np.array(seen)
qq = quota_queue([S['правила'], *sub_sc], k); QT = pd.Series(types[qq]).replace('', 'норма').value_counts()
print('квотная очередь k=%d: precision %.3f' % (k, y[qq].mean()), QT.to_dict()); print('гибридная очередь: ', pd.Series(types[top]).replace('', 'норма').value_counts().to_dict())
ALERT = Q.iloc[0][['entry_id','user','account','amount','reasons']].to_dict()
PROMPT = ('Роль: помощник аудитора. Опиши алерт ТОЛЬКО по полям ниже, ничего не добавляй и не оценивай виновность.\n'
          'Формат: 1) что необычно (2 предложения) 2) какие данные проверить (3 пункта).\n'
          f'Поля алерта: {ALERT}\nЕсли данных недостаточно — напиши «недостаточно данных».')
print(PROMPT)   # отправляется только в локальную/разрешённую модель; ответ проверяется по полям алерта

## 5. Журнал событий: окна по 40 строк без разметки
Матрица «окно × шаблон» → Isolation Forest. Учебная демонстрация механики: разметки нет, метрик нет, результат — окна на ручной разбор. В HDFS_2k блоки почти не повторяются, поэтому сессионный анализ делают на полном HDFS_v1 из Loghub.

In [ ]:
URL = 'https://raw.githubusercontent.com/logpai/loghub/master/HDFS/HDFS_2k.log'
try:
    raw = urllib.request.urlopen(URL, timeout=20).read().decode().splitlines()
    rx = re.compile(r'^(\d{6}) (\d{6}) (\d+) (\w+) ([\w.$]+): (.*)$'); L = [m.groups() for m in map(rx.match, raw) if m]
    lg = pd.DataFrame(L, columns=['date','time','pid','level','component','msg'])
    lg['tpl'] = lg.msg.str.replace(r'blk_-?\d+','<BLK>',regex=True).str.replace(r'\d+\.\d+\.\d+\.\d+(:\d+)?','<IP>',regex=True).str.replace(r'\d+','<N>',regex=True)
    M = pd.crosstab(np.arange(len(lg))//40, lg.tpl); sc_ = -IsolationForest(n_estimators=200, random_state=SEED).fit(M).score_samples(M)
    print('окон', len(M), '| шаблонов', M.shape[1]); print(pd.Series(sc_, M.index).sort_values(ascending=False).head(3))
except Exception as e: print('Loghub недоступен:', e)

In [ ]:
json.dump({'methods': R, 'recall_at_500_by_type': RT.to_dict('index'), 'IF_P200_seeds_mean': float(np.mean(P200)), 'IF_P200_seeds_std': float(np.std(P200)),
           'IF_P200_by_ref_size': SZ, 'queue_precision_k100': float(Q.atype.ne('').mean()), 'quota_queue_precision_k100': float(y[qq].mean()), 'quota_queue_types': QT.to_dict()}, open('p2_metrics.json','w'), indent=1, ensure_ascii=False)

## Задания
1. Уберите признаки `pair_share` и `user_day_cnt`: как меняется полнота по типам D и E?
2. Подберите вес в `гибрид` (сейчас правила идут строго первыми): сделайте мягкое смешивание и сравните P@200.
3. Выберите k под ёмкость 100 проводок в неделю и посчитайте p·V > c для своих V и c.
4. Проверьте вручную 30 причин срабатывания: в скольких причины подтверждаются данными алерта?
5. Найдите в Loghub две самые необычные сессии и опишите, что проверит аудитор.